# SpikeDistill-RUL baseline suite

Select a GPU runtime and run the cells in order. The default sweep covers FD001–FD004. Each neural model gets the same fixed training budget and is selected by capped validation RMSE. Predictions are inverse-scaled to engine cycles for every reported metric. Keep official test evaluation disabled until the protocol and model choices are frozen.

The notebook writes per-engine predictions, per-seed metrics and checkpoints to Drive. Use one runtime per run name.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import torch
from google.colab import drive

drive.mount("/content/drive")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime for the full sweep.")
print(torch.cuda.get_device_name(0))

project = Path("/content/drive/MyDrive/ML projec")
raw = project / "CMaps"
assert raw.is_dir(), raw
subsets = ["FD001", "FD002", "FD003", "FD004"]
seeds = ["42", "43", "44"]
epochs = "30"
run_name = "baselines_v1"  # Use a new name if changing code or training settings.
output = project / "SpikeDistill-RUL" / "colab_runs" / run_name
output.mkdir(parents=True, exist_ok=True)
repo = Path("/content/ML3103-project")
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", "https://github.com/abhav-arora/ML3103-project.git", str(repo)], check=True)

revision_file = output / "git_revision.txt"
if revision_file.exists():
    revision = revision_file.read_text().strip()
    subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True)
else:
    subprocess.run(["git", "fetch", "origin", "main"], cwd=repo, check=True)
    revision = subprocess.check_output(["git", "rev-parse", "FETCH_HEAD"], cwd=repo, text=True).strip()
subprocess.run(["git", "checkout", "--detach", revision], cwd=repo, check=True)
legacy = (repo / "run_baselines.py").is_file()
assert legacy or (repo / "spikedistill/training/baselines.py").is_file(), "Baseline code is missing from the selected commit."
prepare_entry = ["prepare_data.py"] if legacy else ["-m", "spikedistill.data.prepare"]
baseline_entry = ["run_baselines.py"] if legacy else ["-m", "spikedistill.training.baselines"]
revision_file.write_text(revision + "\n")
print("Git revision", revision)

In [ ]:
test_names = ["test_baselines", "test_metrics", "test_prepare_data"]
if not legacy:
    test_names = [f"tests.{name}" for name in test_names]
subprocess.run([sys.executable, "-m", "unittest", *test_names], cwd=repo, check=True)
data_root = Path("/content/spikedistill_baselines_data")
subprocess.run([sys.executable, *prepare_entry, "--data-dir", str(raw),
                "--output-dir", str(data_root), "--subsets", *subsets], cwd=repo, check=True)

command = [sys.executable, *baseline_entry, "--data-root", str(data_root), "--output", str(output),
           "--subsets", *subsets, "--seeds", *seeds, "--epochs", epochs, "--device", "cuda"]
with subprocess.Popen([*command, "--phase", "train"], cwd=repo,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, process.args)

In [ ]:
import pandas as pd

def show_metrics(filename):
    rows = json.loads((output / filename).read_text())
    table = []
    for row in rows:
        values = {key: row[key] for key in ("subset", "model", "target", "seeds")}
        for metric in ("MAE", "MSE", "RMSE", "R2", "cmapss_score"):
            stats = row[metric]
            values[metric] = stats["mean"] if stats else None
            values[metric + "_sd"] = stats["sample_sd"] if stats else None
        table.append(values)
    display(pd.DataFrame(table))

show_metrics("train_summary.json")

## Official test
Enable the next cell only after you freeze the protocol. The same cycle-scale predictions are scored against raw official RUL and capped RUL separately. Use raw RUL for the official comparison and identify capped RUL clearly when reporting it.

In [ ]:
RUN_OFFICIAL_TEST = False
if RUN_OFFICIAL_TEST:
    subprocess.run([*command, "--phase", "test"], cwd=repo, check=True)
    show_metrics("test_summary.json")